# Advice Agent
adviceAgent.py is an implementation of baseAgent.py that uses tools/adviceApiTool.py
Its general process is to:
 1. Follow the system prompt given.
 2. Receive a prompt relating to healthcare, that could have personal health parameters (age, sex, etc.)
 3. Parse the parameters from the prompt (if any) and utilize the API tool.
 4. Use the output from the API tool to provide an answer to the user and terminate.

## Imports
NOTE: We are using AutoGen version 0.9.10. This is before UserAgent, which has a very different process.

In [ ]:
from autogen import LLMConfig, UserProxyAgent
from baseAgent import BaseAgent
from tools.adviceApiTool import AdviceAPITool

## AdviceAgent Class
This focuses on giving the agent a good prompt.
This prompt was not given security, it just focuses on functionality for the time being.
The conversation protocol was implemented to prevent accidental delegations from wasting too much time.

In [ ]:
class AdviceAgent(BaseAgent):
    def __init__(self):
        self.name = "AdviceAgent"
        self.system_message = """
            You are a healthcare professional, specializing in giving users advice. When prompted, you must use the AdviceAPITool tool to answer any question
            about healthcare tips and advice. Do not answer directly. Always use the tool first.
            Use the url https://odphp.health.gov/myhealthfinder/api/v4/myhealthfinder.json? as the baseUrl for the AdviceAPITool.
            When you use the tool, only return the output of the tool.
            After providing the answer, say 'TERMINATE' to end the conversation.
            
            "CONVERSATION PROTOCOL:"
            "1. Only speak when directly asked a question or when you have the specific information requested."
            "2. If another agent can handle the query, stay silent."
            "3. Default to silence unless you're certain your input is needed."
            """
        self.tools = [AdviceAPITool()]
        
        super().__init__(
            name = self.name,
            system_message = self.system_message,
            tools = self.tools
        )

## Example Run
Here a user proxy agent is defined, which is the core communicator with the user.
For the tools to work properly, `human_input_mode` must be set to `TERMINATE` or `NEVER`.

Then we initialize our advice agent and register execution with the user proxy agent.
The user proxy agent also needs to get the LLM config that was explained in baseAgent.ipynb

The conversation here is just the user proxy and the advice agent, so we use initialize chat.
The first prompt will be the one written in `message=` but after the auto replies which utilize the tool, the terminal is active for further messages.

In [ ]:
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply= 1,
    code_execution_config={"use_docker": False},
)

AdviceAgent = AdviceAgent()

AdviceAgent.registerExecution(user_proxy)

config = LLMConfig.from_json(path = "OAI_CONFIG_LIST.json")

user_proxy.initiate_chat(
    AdviceAgent.agent,
    message="I would like to get some healthcare advice. I am a 35 year old female, who is not pregnant, I am sexually active, and I do not smoke tobacco.",
    llm_config=config
)
